# FeederBW Time Series Foundation Models (TSFM) – Tutorial

This notebook serves as a start for exploring time series forcasting of low voltage power grid data with time series foundation models. Feel free to: 
- Run the complete notebook and look at the outcomes
- Extend the notebook code (see also [tutorial.ipynb](tutorial.ipynb)) as you like. 
- Try out the challenges at the end of the notebook. 

**Dataset**: 200 low-voltage distribution feeders in Baden-Württemberg, Germany.  
1-minute resolution measurements, 2023-04 to 2025-03.  
Nature Scientific Data: [Website scientific data](https://www.nature.com/articles/s41597-026-08152-9)

## 1. Setup

The following code is used to make the necessary installations in Google Colab.

In [ ]:
!git clone --branch ai-grids-tsfm-workshop-2026 --single-branch https://github.com/KIT-IAI/FeederBW.git
%cd /content/FeederBW
!uv sync
# !pip install zenodo-get

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.download.download import download_data
from src.load.load import load_data
from src.model.forecast import forecast
from src.model.pseudo_measurement_generation import pseudo_measurement_generation

## 2. Downloading the Dataset

The dataset is hosted on [Zenodo](https://zenodo.org/records/17831177).
You can download all files or selective parts using `download_data()`.

For this tutorial we download only metadata and weather. Measurement data was added to the repository already. 

In [ ]:
# Skip if you already have the data locally
download_data(files=["feeder_metadata", "weather_data"], additive=True)
# NOTE (manuel): the measurement data is already available in Git for convenience during the workshop (unzip the files takes too much time in Google Colab)

## 3. Loading Data

`load_data()` supports flexible loading with:
- **`feeders`** – list of feeder IDs (1–200)
- **`columns`** – column selection (measurement, weather, metadata columns can be mixed; they are automatically joined)
- **`time_period`** – `(start, end)` as ISO strings or datetime objects
- **`temporal_resolution`** – resample to `"1min"`, `"5min"`, `"15min"`, `"30min"`, `"1h"`, `"1d"`, etc.
- **`aggregation_type`** – per-column aggregation (mean/sum/min/max) used when downsampling
- **`interpolation_type`** – per-column interpolation (linear/forward/backward) used when upsampling

### 3.1 Load measurement data only

In [ ]:
meas = load_data(
    feeders=[37, 33], # 97
    columns=["feeder", "timestamp_UTC", "active_power_kW", "apparent_power_kVA"],
    time_period=("2024-01-01T00:00:00", "2024-02-01T00:00:00"),
)
print(f"Shape: {meas.shape}")
meas.head()

### 3.2 Resample to hourly resolution

When specifying `temporal_resolution`, you can control per-column aggregation via `aggregation_type`.

In [ ]:
meas_hourly = load_data(
    feeders=[37, 33],
    columns=["feeder", "timestamp_UTC", "active_power_kW"],
    time_period=("2024-01-01T00:00:00", "2024-02-01T00:00:00"),
    temporal_resolution="1h",
    aggregation_type={"active_power_kW": "mean"},
)
print(f"Shape: {meas_hourly.shape}")
meas_hourly.head()

### 3.3 Join measurements with weather and metadata

Passing columns from different sources causes `load_data()` to automatically join them on `feeder` and `timestamp_UTC`.

In [ ]:
joined = load_data(
    feeders=[37, 33],
    columns=[
        "feeder", "timestamp_UTC",
        "active_power_kW",                   # measurement
        "air_temperature_C",                 # weather
        "housing_units_count",               # metadata
    ],
    time_period=("2024-06-01T00:00:00", "2024-07-01T00:00:00"),
    temporal_resolution="1h",
    aggregation_type={"active_power_kW": "mean", "air_temperature_C": "mean"},
)
print(f"Shape: {joined.shape}")
joined.head(10)

## 4. Exploring the Data

Let's visualise active power and air temperature for one feeder over a few days.

In [ ]:
Feeder_37 = joined.filter(pl.col("feeder") == 37).sort("timestamp_UTC")
ts = Feeder_37["timestamp_UTC"].to_list()
power = Feeder_37["active_power_kW"].to_list()
temp = Feeder_37["air_temperature_C"].to_list()

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(ts, power, color="steelblue", label="Active power (kW)")
ax1.set_ylabel("Active power (kW)", color="steelblue")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

ax2 = ax1.twinx()
ax2.plot(ts, temp, color="tomato", linestyle="--", alpha=0.7, label="Air temperature (°C)")
ax2.set_ylabel("Air temperature (°C)", color="tomato")
ax2.tick_params(axis="y", labelcolor="tomato")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
ax1.set_title("Feeder 33 – Active power vs. air temperature (June 2024)")
fig.tight_layout()
plt.show()

### 4.1 Daily load profiles

Aggregate to 15-minute resolution and plot the median daily profile for Feeder 33.

In [ ]:
meas_15min = load_data(
    feeders=[33],
    columns=["feeder", "timestamp_UTC", "active_power_kW"],
    time_period=("2024-01-01T00:00:00", "2024-04-01T00:00:00"),
    temporal_resolution="15min",
    aggregation_type={"active_power_kW": "mean"},
)

daily_profile = (
    meas_15min
    .with_columns(pl.col("timestamp_UTC").dt.time().alias("time_of_day"))
    .group_by("time_of_day")
    .agg(
        pl.col("active_power_kW").quantile(0.05).alias("q05"),
        pl.col("active_power_kW").median().alias("median"),
        pl.col("active_power_kW").quantile(0.95).alias("q95"),
    )
    .sort("time_of_day")
)

times = [str(t) for t in daily_profile["time_of_day"].to_list()]
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(times, daily_profile["q05"].to_list(), daily_profile["q95"].to_list(),
                alpha=0.3, color="steelblue", label="5–95th percentile")
ax.plot(times, daily_profile["median"].to_list(), color="steelblue", label="Median")
ax.set_xticks(range(0, len(times), 8))
ax.set_xticklabels(times[::8], rotation=45)
ax.set_xlabel("Time of day")
ax.set_ylabel("Active power (kW)")
ax.set_title("Feeder 33 – Daily load profile (Q1 2024, 15-min bins)")
ax.legend()
fig.tight_layout()
plt.show()

### 4.2 Feeder metadata overview

The metadata CSV contains information about installed DER capacity, consumer types, and load profiles for each feeder.

In [ ]:
meta = pl.read_csv("data/zenodo/feeder_metadata.csv")
print(f"Shape: {meta.shape}")
print(meta.columns)
meta.head()

In [ ]:
# Distribution of installed PV capacity across feeders
pv_stats = meta.group_by("feeder").agg(pl.col("PV_systems_kW").mean()).sort("feeder")

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(pv_stats["feeder"].to_list(), pv_stats["PV_systems_kW"].to_list(), color="goldenrod", width=0.8)
ax.set_xlabel("Feeder ID")
ax.set_ylabel("Avg. PV capacity (kW)")
ax.set_title("Installed PV capacity per feeder")
fig.tight_layout()
plt.show()

## 5. Forecasting

### 5.1 Naive Forecast

`forecast()` implements a **naive quantile baseline**: for each test day (from `split_date` onwards), the 5th, 50th, and 95th percentile of `target_column` are computed for each `(feeder, time_of_day)` combination using all historically available data.

Metrics returned (per feeder + average):
- **MAE** – Mean Absolute Error on the median forecast
- **Pinball loss** – for each of the three quantiles (q05, q50, q95)
- **Winkler score** – 90% prediction interval quality (lower is better)

In [ ]:
data_for_forecast = load_data(
    feeders=[37],
    columns=["feeder", "timestamp_UTC", "active_power_kW"],
    time_period=("2024-01-01T00:00:00", "2024-04-01T00:00:00"),
    temporal_resolution="1h",
    aggregation_type={"active_power_kW": "mean"},
)

forecast_metrics = forecast(
    data=data_for_forecast,
    split_date="2024-03-01",
    target_column="active_power_kW",
)
forecast_metrics

In [ ]:
# NOTE (manuel): the subfigures are created randomly from the forecast results. 
from IPython.display import Image
Image("data/forecast_figure.png")

### 5.2 Chronos-2 Forecast

In [ ]:
# TODO

## 6. Challenges